# 02 · Data Model

**Goal:** turn the five raw files into clean tables that every later analysis reads from.

| Layer | Table | Grain |
|---|---|---|
| Staging | `stg_players`, `stg_cash`, `stg_tourn`, `stg_deposits`, `stg_withdrawals` | Cleaned copies of the raw files |
| Facts | `fct_player_day` | One row per player per active day |
| Facts | `fct_player_week` | One row per player per active week (weeks start Monday) |
| Dimension | `dim_player` | One row per player with first and last activity and lifetime totals |

**Analysis window:** 2015-02-01 to 2020-06-30. The SQL is written in SQLite and kept close to standard SQL.

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
con = sqlite3.connect(ROOT / "data" / "poker.db")
END_DATE = "2020-06-30"

def q(sql):
    return pd.read_sql_query(sql, con)

def run(sql):
    con.executescript(sql)
    con.commit()

## 1. Players and markets

Country codes are ISO 3166 numeric. The ten largest countries are kept as their own market; the rest are grouped.

In [2]:
COUNTRIES = {
    276: "Germany", 250: "France", 724: "Spain", 56: "Belgium", 826: "United Kingdom", 40: "Austria",
    203: "Czechia", 756: "Switzerland", 348: "Hungary", 528: "Netherlands", 703: "Slovakia", 616: "Poland",
    788: "Tunisia", 643: "Russia", 752: "Sweden", 442: "Luxembourg", 124: "Canada", 484: "Mexico",
    70: "Bosnia and Herzegovina", 31: "Azerbaijan", 372: "Ireland", 578: "Norway", 784: "United Arab Emirates",
    638: "Reunion", 8: "Albania", 862: "Venezuela", 470: "Malta", 458: "Malaysia", 398: "Kazakhstan",
    352: "Iceland", 818: "Egypt", 604: "Peru", 504: "Morocco", 360: "Indonesia", 50: "Bangladesh",
    858: "Uruguay", 634: "Qatar", 498: "Moldova", 392: "Japan", 268: "Georgia", 254: "French Guiana",
    795: "Turkmenistan", 764: "Thailand", 682: "Saudi Arabia", 600: "Paraguay", 554: "New Zealand",
    524: "Nepal", 496: "Mongolia", 492: "Monaco", 480: "Mauritius", 474: "Martinique", 417: "Kyrgyzstan",
    312: "Guadeloupe", 188: "Costa Rica", 68: "Bolivia", 20: "Andorra", 12: "Algeria",
}
TOP_MARKETS = [276, 250, 724, 56, 826, 40, 203, 756, 348, 528]
OTHER_EUROPE = [703, 616, 643, 752, 442, 70, 372, 578, 8, 470, 352, 498, 492, 20, 268, 31]

def market(code):
    if pd.isna(code):
        return "Unknown"
    code = int(code)
    if code in TOP_MARKETS:
        return COUNTRIES[code]
    return "Other Europe" if code in OTHER_EUROPE else "Rest of World"

countries = pd.DataFrame({"country_id": list(COUNTRIES), "country_name": list(COUNTRIES.values())})
countries["market"] = countries["country_id"].map(market)
countries.to_sql("ref_country", con, if_exists="replace", index=False)

run("""
drop table if exists stg_players;
create table stg_players as
select d.UserID              as player_id,
       d.SystemAgeAsOfReg    as age_at_registration,
       d.Gender              as gender,
       cast(d.CountryID as integer) as country_id,
       coalesce(c.country_name, 'Unknown') as country_name,
       coalesce(c.market, 'Unknown')       as market,
       case when d.SystemAgeAsOfReg < 25 then '18-24'
            when d.SystemAgeAsOfReg < 35 then '25-34'
            when d.SystemAgeAsOfReg < 45 then '35-44'
            when d.SystemAgeAsOfReg < 55 then '45-54'
            else '55+' end   as age_band
from raw_demographics d
left join ref_country c on c.country_id = cast(d.CountryID as integer);
""")
q("select market, count(*) as players from stg_players group by 1 order by 2 desc")

,market,players
0,Germany,1416
1,France,1252
2,Other Europe,383
3,Spain,336
4,Belgium,300
5,United Kingdom,231
6,Austria,216
7,Czechia,190
8,Switzerland,185
9,Hungary,184


## 2. Poker activity

Dates are restricted to the analysis window. There are no duplicate player-days or all-zero rows (see notebook 01), so no other cleaning is needed.

In [3]:
run(f"""
drop table if exists stg_cash;
create table stg_cash as
select UserID as player_id, Date as activity_date,
       Windows as cash_sessions, StakesC as cash_stakes, WinningsC as cash_winnings
from raw_cash where Date <= '{END_DATE}';

drop table if exists stg_tourn;
create table stg_tourn as
select UserID as player_id, Date as activity_date,
       Trnmnts as tournaments, StakesT as tourn_spend, WinningsT as tourn_prizes
from raw_tourn where Date <= '{END_DATE}';
""")
q("select 'cash' as source, count(*) as rows, count(distinct player_id) as players from stg_cash union all select 'tournaments', count(*), count(distinct player_id) from stg_tourn")

,source,rows,players
0,cash,51666,4229
1,tournaments,82304,3446


## 3. Payments

Cleaning rules, following the original authors' R code:

- Remove exact duplicates (same player, dates, time, method, category, card type, amount and status); keep the lowest transaction ID.
- Standardize payment categories (`ewallet` becomes `eWallet`) and card types.
- A transaction belongs to the date the player requested it (`SummaryDate`). Requests processed after the window ends are excluded.
- Deposit status `S` is completed and `F` is failed. Withdrawal status `S` is completed and `R` is reversed by the player.

In [4]:
def payment_sql(src, id_col, out):
    return f"""
drop table if exists {out};
create table {out} as
with ranked as (
    select *, row_number() over (
        partition by UserID, SummaryDate, ProcessDate, ProcessTime, PayMeth, PayMethCat, CardType, Amount, Status
        order by {id_col}) as dup_rank
    from {src}
)
select {id_col} as txn_id,
       UserID   as player_id,
       SummaryDate as request_date,
       ProcessDate as process_date,
       ProcessTime as process_time,
       PayMeth  as pay_method,
       case PayMethCat when 'ewallet' then 'eWallet'
                       when 'Cards' then 'Other card'
                       when 'Others' then 'Other'
                       else PayMethCat end as pay_category,
       case upper(trim(coalesce(CardType, '')))
            when 'CCD' then 'Credit'
            when 'DCD' then 'Debit'
            when 'DD'  then 'Debit'
            when 'PP'  then 'Prepaid'
            else 'Undetermined' end as card_type,
       Amount   as amount,
       Status   as status,
       case when Status = 'S' then 1 else 0 end as is_completed
from ranked
where dup_rank = 1 and ProcessDate <= '{END_DATE}';
"""

run(payment_sql("raw_deposits", "DepositID", "stg_deposits"))
run(payment_sql("raw_withdrawals", "WithdrawalID", "stg_withdrawals"))
q("""
select 'deposits' as source, count(*) as rows, sum(is_completed) as completed, count(distinct player_id) as players from stg_deposits
union all select 'withdrawals', count(*), sum(is_completed), count(distinct player_id) from stg_withdrawals
""")

,source,rows,completed,players
0,deposits,293424,222394,5008
1,withdrawals,32095,17107,1826


In [5]:
q("select pay_category, count(*) as attempts, round(100.0 * avg(is_completed), 1) as success_pct from stg_deposits group by 1 order by 2 desc")

,pay_category,attempts,success_pct
0,EEA debit/credit card,108227,72.4
1,eWallet,72997,70.6
2,Bank Transfer,58410,75.9
3,Voucher,53102,89.8
4,Prepaid Cards,318,77.7
5,Other card,257,69.3
6,Mobile Carrier Billing,110,7.3
7,Other,3,0.0


## 4. Player-day fact table

One row per player per day with any poker or payment activity. Key measures:

- `poker_active`: 1 if the player played a cash game or tournament that day
- `player_result`: cash winnings plus tournament prizes minus stakes and spend (positive means the player came out ahead)
- `net_deposits`: completed deposits minus completed withdrawals, the player-value measure used throughout

In [6]:
run("""
drop table if exists fct_player_day;
create table fct_player_day as
with keys as (
    select player_id, activity_date from stg_cash
    union select player_id, activity_date from stg_tourn
    union select player_id, request_date from stg_deposits
    union select player_id, request_date from stg_withdrawals
),
dep as (
    select player_id, request_date as activity_date,
           count(*) as deposit_attempts,
           sum(is_completed) as deposits_completed,
           sum(1 - is_completed) as deposits_failed,
           sum(case when is_completed = 1 then amount else 0 end) as deposit_amount
    from stg_deposits group by 1, 2
),
wd as (
    select player_id, request_date as activity_date,
           count(*) as withdrawal_requests,
           sum(case when is_completed = 1 then amount else 0 end) as withdrawal_amount,
           sum(case when status = 'R' then amount else 0 end) as withdrawal_reversed_amount
    from stg_withdrawals group by 1, 2
)
select k.player_id,
       k.activity_date,
       date(k.activity_date, 'weekday 0', '-6 days') as week_start,
       substr(k.activity_date, 1, 7) || '-01'        as month_start,
       case when c.player_id is not null or t.player_id is not null then 1 else 0 end as poker_active,
       coalesce(c.cash_sessions, 0)  as cash_sessions,
       coalesce(c.cash_stakes, 0)    as cash_stakes,
       coalesce(c.cash_winnings, 0)  as cash_winnings,
       coalesce(t.tournaments, 0)    as tournaments,
       coalesce(t.tourn_spend, 0)    as tourn_spend,
       coalesce(t.tourn_prizes, 0)   as tourn_prizes,
       coalesce(c.cash_winnings, 0) + coalesce(t.tourn_prizes, 0)
         - coalesce(c.cash_stakes, 0) - coalesce(t.tourn_spend, 0) as player_result,
       coalesce(dep.deposit_attempts, 0)   as deposit_attempts,
       coalesce(dep.deposits_completed, 0) as deposits_completed,
       coalesce(dep.deposits_failed, 0)    as deposits_failed,
       coalesce(dep.deposit_amount, 0)     as deposit_amount,
       coalesce(wd.withdrawal_requests, 0) as withdrawal_requests,
       coalesce(wd.withdrawal_amount, 0)   as withdrawal_amount,
       coalesce(wd.withdrawal_reversed_amount, 0) as withdrawal_reversed_amount,
       coalesce(dep.deposit_amount, 0) - coalesce(wd.withdrawal_amount, 0) as net_deposits
from keys k
left join stg_cash  c   on c.player_id = k.player_id   and c.activity_date = k.activity_date
left join stg_tourn t   on t.player_id = k.player_id   and t.activity_date = k.activity_date
left join dep           on dep.player_id = k.player_id and dep.activity_date = k.activity_date
left join wd            on wd.player_id = k.player_id  and wd.activity_date = k.activity_date;

create index idx_fpd on fct_player_day(player_id, activity_date);
""")
q("select count(*) as rows, count(distinct player_id) as players, min(activity_date) as first_day, max(activity_date) as last_day, sum(poker_active) as poker_days from fct_player_day")

,rows,players,first_day,last_day,poker_days
0,233742,5027,2015-02-01,2020-06-30,113561


## 5. Player-week fact table

The weekly report (notebook 03) reads from this table. Weeks start on Monday.

In [7]:
run("""
drop table if exists fct_player_week;
create table fct_player_week as
select player_id, week_start,
       max(poker_active)       as poker_active,
       sum(poker_active)       as poker_days,
       sum(cash_sessions)      as cash_sessions,
       sum(tournaments)        as tournaments,
       sum(cash_stakes + tourn_spend) as total_stakes,
       sum(player_result)      as player_result,
       sum(deposit_attempts)   as deposit_attempts,
       sum(deposits_completed) as deposits_completed,
       sum(deposits_failed)    as deposits_failed,
       sum(deposit_amount)     as deposit_amount,
       sum(withdrawal_amount)  as withdrawal_amount,
       sum(withdrawal_reversed_amount) as withdrawal_reversed_amount,
       sum(net_deposits)       as net_deposits
from fct_player_day
group by 1, 2;
""")
q("select count(*) as rows, count(distinct week_start) as weeks, min(week_start) as first_week, max(week_start) as last_week from fct_player_week")

,rows,weeks,first_week,last_week
0,94732,284,2015-01-26,2020-06-29


## 6. Player dimension

First and last activity dates, the first deposit method, and lifetime totals. `first_play_month` defines the acquisition cohort, because every player registered in February 2015.

In [8]:
run("""
drop table if exists dim_player;
create table dim_player as
with play as (
    select player_id, min(activity_date) as first_play_date, max(activity_date) as last_play_date,
           count(*) as poker_days
    from fct_player_day where poker_active = 1 group by 1
),
first_dep as (
    select player_id, pay_method as first_deposit_method, pay_category as first_deposit_category,
           request_date as first_deposit_date
    from (select *, row_number() over (partition by player_id order by request_date, process_time, txn_id) as rn
          from stg_deposits where is_completed = 1)
    where rn = 1
),
first_attempt as (
    select player_id, status as first_attempt_status
    from (select *, row_number() over (partition by player_id order by request_date, process_time, txn_id) as rn
          from stg_deposits)
    where rn = 1
),
totals as (
    select player_id, sum(deposit_amount) as lifetime_deposits, sum(withdrawal_amount) as lifetime_withdrawals,
           sum(net_deposits) as lifetime_net_deposits, sum(cash_stakes + tourn_spend) as lifetime_stakes,
           sum(deposits_failed) as lifetime_failed_deposits
    from fct_player_day group by 1
)
select p.*,
       pl.first_play_date, pl.last_play_date,
       substr(pl.first_play_date, 1, 7) || '-01' as first_play_month,
       coalesce(pl.poker_days, 0) as poker_days,
       fd.first_deposit_date, fd.first_deposit_method, fd.first_deposit_category,
       fa.first_attempt_status,
       coalesce(t.lifetime_deposits, 0)     as lifetime_deposits,
       coalesce(t.lifetime_withdrawals, 0)  as lifetime_withdrawals,
       coalesce(t.lifetime_net_deposits, 0) as lifetime_net_deposits,
       coalesce(t.lifetime_stakes, 0)       as lifetime_stakes,
       coalesce(t.lifetime_failed_deposits, 0) as lifetime_failed_deposits
from stg_players p
left join play pl         on pl.player_id = p.player_id
left join first_dep fd    on fd.player_id = p.player_id
left join first_attempt fa on fa.player_id = p.player_id
left join totals t        on t.player_id = p.player_id;
""")
q("select count(*) as players, sum(first_play_date is null) as never_played, sum(first_deposit_date is null) as never_deposited_successfully from dim_player")

,players,never_played,never_deposited_successfully
0,5028,3,22


## 7. QA: reconcile the model back to the source

Every total in the fact tables should match the staging tables exactly. If any row shows a difference, the model has a join or filter bug.

In [9]:
qa = q("""
select 'cash stakes' as measure,
       (select round(sum(cash_stakes), 2) from stg_cash) as source,
       (select round(sum(cash_stakes), 2) from fct_player_day) as model
union all select 'tournament spend', (select round(sum(tourn_spend), 2) from stg_tourn), (select round(sum(tourn_spend), 2) from fct_player_day)
union all select 'completed deposits', (select round(sum(amount), 2) from stg_deposits where is_completed = 1), (select round(sum(deposit_amount), 2) from fct_player_day)
union all select 'failed deposit count', (select count(*) from stg_deposits where is_completed = 0), (select sum(deposits_failed) from fct_player_day)
union all select 'completed withdrawals', (select round(sum(amount), 2) from stg_withdrawals where is_completed = 1), (select round(sum(withdrawal_amount), 2) from fct_player_day)
union all select 'net deposits (week vs day)', (select round(sum(net_deposits), 2) from fct_player_day), (select round(sum(net_deposits), 2) from fct_player_week)
union all select 'net deposits (player vs day)', (select round(sum(net_deposits), 2) from fct_player_day), (select round(sum(lifetime_net_deposits), 2) from dim_player)
union all select 'players', (select count(*) from stg_players), (select count(*) from dim_player)
""")
qa["difference"] = (qa["model"] - qa["source"]).round(2)
assert (qa["difference"] == 0).all(), "Model does not reconcile to source"
qa

,measure,source,model,difference
0,cash stakes,29044128.82,29044128.82,0.0
1,tournament spend,3621875.26,3621875.26,0.0
2,completed deposits,9602520.19,9602520.19,0.0
3,failed deposit count,71030.00,71030.00,0.0
4,completed withdrawals,4371204.62,4371204.62,0.0
5,net deposits (week vs day),5231315.58,5231315.58,0.0
6,net deposits (player vs day),5231315.58,5231315.58,0.0
7,players,5028.00,5028.00,0.0


Note: 2015-02-01 is a Sunday and 2020-06-30 is a Tuesday, so the first and last weeks are partial. Reports that compare weeks use only full weeks, from 2015-02-02 to 2020-06-28.

All measures reconcile. The model is ready for the weekly report (03), cohorts and LTV (04), payments (05), and monitoring (06).

In [10]:
con.close()